# Building Real Architectures from Scratch

**What you'll learn in this notebook:**
- **ResNet**: Skip connections, BasicBlock, build ResNet-18
- **GPT**: Causal attention, autoregressive generation
- **ViT**: Patch embedding, position embedding, classification
- Comparing architectures: parameter counts, FLOPs, and use cases

**Prerequisites:** Understanding of convolutions, attention mechanisms, and basic training loops.

**All code runs on CPU — no GPU required.**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
print(f"PyTorch version: {torch.__version__}")

In [ ]:
def count_params(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def format_params(n):
    if n >= 1e9: return f"{n/1e9:.1f}B"
    if n >= 1e6: return f"{n/1e6:.1f}M"
    if n >= 1e3: return f"{n/1e3:.1f}K"
    return str(n)

## Part 1: ResNet — Learning Residual Functions

**Key insight:** Very deep networks are hard to train because gradients vanish. ResNet solves this with **skip connections**: instead of learning $H(x)$, learn the residual $F(x) = H(x) - x$, then output $F(x) + x$.

This makes it easy to learn the identity (just set $F(x) = 0$), allowing gradients to flow freely through the skip connection.

In [ ]:
class BasicBlock(nn.Module):
    """ResNet BasicBlock: two 3x3 convolutions with a skip connection."""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # If dimensions change, need a projection for the skip connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
    
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual  # The skip connection!
        out = F.relu(out)
        return out


# Test BasicBlock
block = BasicBlock(64, 64)
x = torch.randn(1, 64, 32, 32)
print(f"Same dimensions: {x.shape} → {block(x).shape}")

block_down = BasicBlock(64, 128, stride=2)
print(f"Downsampling:    {x.shape} → {block_down(x).shape}")

### Why Skip Connections Work

Without skip connections, gradients must flow through every layer — they can vanish or explode. With skip connections, gradients have a "highway" directly through the network:

$\frac{\partial \text{loss}}{\partial x} = \frac{\partial \text{loss}}{\partial (F(x) + x)} = \frac{\partial \text{loss}}{\partial F(x)} \cdot \frac{\partial F(x)}{\partial x} + \frac{\partial \text{loss}}{\partial x}$

The $+1$ term means gradients always flow, even if $F(x)$ has vanishing gradients.

In [ ]:
# Demonstrate: gradient flow with and without skip connections
class PlainBlock(nn.Module):
    """Block WITHOUT skip connection."""
    def __init__(self, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, 3, padding=1)
        self.conv2 = nn.Conv2d(dim, dim, 3, padding=1)
    
    def forward(self, x):
        return torch.relu(self.conv2(torch.relu(self.conv1(x))))

class ResidualBlock(nn.Module):
    """Block WITH skip connection."""
    def __init__(self, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, 3, padding=1)
        self.conv2 = nn.Conv2d(dim, dim, 3, padding=1)
    
    def forward(self, x):
        return torch.relu(self.conv2(torch.relu(self.conv1(x))) + x)

# Stack many blocks and check gradient magnitude
def check_gradient_flow(block_cls, num_blocks=20, dim=16):
    blocks = nn.Sequential(*[block_cls(dim) for _ in range(num_blocks)])
    x = torch.randn(1, dim, 8, 8, requires_grad=True)
    output = blocks(x)
    output.sum().backward()
    return x.grad.abs().mean().item()

plain_grad = check_gradient_flow(PlainBlock, num_blocks=20)
resid_grad = check_gradient_flow(ResidualBlock, num_blocks=20)
print(f"Gradient magnitude through 20 blocks:")
print(f"  Plain (no skip):  {plain_grad:.6f}")
print(f"  Residual (skip):  {resid_grad:.6f}")
print(f"  Ratio: {resid_grad/plain_grad:.1f}x stronger gradients with skip connections!")

In [ ]:
class ResNet18(nn.Module):
    """ResNet-18: 4 stages of BasicBlocks, global average pool, classifier."""
    
    def __init__(self, num_classes=1000):
        super().__init__()
        # Initial convolution (7x7, stride 2) + maxpool
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        
        # 4 stages, each with 2 BasicBlocks
        self.stage1 = self._make_stage(64, 64, num_blocks=2, stride=1)
        self.stage2 = self._make_stage(64, 128, num_blocks=2, stride=2)
        self.stage3 = self._make_stage(128, 256, num_blocks=2, stride=2)
        self.stage4 = self._make_stage(256, 512, num_blocks=2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_stage(self, in_ch, out_ch, num_blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.stem(x)         # [B, 64, 56, 56]
        x = self.stage1(x)       # [B, 64, 56, 56]
        x = self.stage2(x)       # [B, 128, 28, 28]
        x = self.stage3(x)       # [B, 256, 14, 14]
        x = self.stage4(x)       # [B, 512, 7, 7]
        x = self.avgpool(x)      # [B, 512, 1, 1]
        x = x.flatten(1)         # [B, 512]
        x = self.fc(x)           # [B, num_classes]
        return x


resnet = ResNet18(num_classes=10)
x = torch.randn(2, 3, 224, 224)  # Standard ImageNet input size
output = resnet(x)

print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in resnet.parameters()):,}")
print(f"\nStage feature map sizes (for input 224x224):")
with torch.no_grad():
    h = resnet.stem(x)
    print(f"  After stem:   {h.shape}")
    h = resnet.stage1(h); print(f"  After stage1: {h.shape}")
    h = resnet.stage2(h); print(f"  After stage2: {h.shape}")
    h = resnet.stage3(h); print(f"  After stage3: {h.shape}")
    h = resnet.stage4(h); print(f"  After stage4: {h.shape}")

## Part 2: GPT — Autoregressive Language Model

GPT uses **causal (masked) self-attention** so each token can only attend to previous tokens. This enables autoregressive generation: predict the next token, append it, repeat.

Key components: token embedding, positional embedding, causal Transformer blocks, language model head (tied with embedding weights).

In [ ]:
# Verify ResNet-18 works end-to-end with a training step
resnet_small = ResNet18(num_classes=10)
optimizer = torch.optim.SGD(resnet_small.parameters(), lr=0.01)

x = torch.randn(4, 3, 224, 224)
target = torch.randint(0, 10, (4,))

# Forward + backward
logits = resnet_small(x)
loss = F.cross_entropy(logits, target)
loss.backward()
optimizer.step()

print(f"Training step successful!")
print(f"  Input: {x.shape}")
print(f"  Logits: {logits.shape}")
print(f"  Loss: {loss.item():.4f}")
print(f"  Predictions: {logits.argmax(dim=1).tolist()}")
print(f"  Targets:     {target.tolist()}")

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_len=512):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        
        # Causal mask (registered as buffer — not a parameter)
        mask = torch.tril(torch.ones(max_len, max_len))
        self.register_buffer("mask", mask.view(1, 1, max_len, max_len))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Project Q, K, V in one shot
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.d_k)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # [3, B, heads, T, d_k]
        
        # Scaled dot-product attention with causal mask
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.proj(out)


class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, max_len=512):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

### GPT Architecture Diagram

```
Input tokens: [The, cat, sat, on]
       ↓
Token Embedding + Position Embedding
       ↓
┌────────────────────────────────────┐ ×N layers
│ LayerNorm → Causal Self-Attention  │
│      + residual connection          │
│ LayerNorm → FFN (MLP)             │
│      + residual connection          │
└────────────────────────────────────┘
       ↓
Final LayerNorm
       ↓
Linear (lm_head, tied with embedding)
       ↓
Logits: probability over vocabulary for each position
```

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_len=512):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, max_len) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        
        # Weight tying: share embedding weights with output projection
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # Tied!
    
    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # [B, T, vocab_size]
        return logits
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive generation."""
        for _ in range(max_new_tokens):
            logits = self(idx)[:, -1, :]  # Last position logits
            probs = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# Build a small GPT
gpt = GPT(vocab_size=256, d_model=128, num_heads=4, num_layers=4, max_len=256)
print(f"GPT parameters: {sum(p.numel() for p in gpt.parameters()):,}")
print(f"  (with weight tying, embedding params are shared with lm_head)")

# Forward pass
tokens = torch.randint(0, 256, (2, 32))
logits = gpt(tokens)
print(f"\nInput: {tokens.shape} → Logits: {logits.shape}")

# Generation
prompt = torch.randint(0, 256, (1, 5))
generated = gpt.generate(prompt, max_new_tokens=10)
print(f"\nGeneration: prompt {prompt.shape} → output {generated.shape}")
print(f"Generated tokens: {generated[0].tolist()}")

## Part 3: Vision Transformer (ViT)

ViT treats an image as a sequence of patches:
1. Split image into fixed-size patches (e.g., 16x16)
2. Linearly embed each patch → a sequence of vectors
3. Prepend a learnable [CLS] token
4. Add positional embeddings
5. Pass through standard Transformer encoder
6. Classify using the [CLS] token's output

### GPT: Parameter Breakdown and Scaling Laws

Where do the parameters live in a GPT model?

In [ ]:
# Break down where GPT parameters live
def gpt_param_breakdown(vocab_size, d_model, num_heads, num_layers):
    """Analyze parameter distribution in a GPT model."""
    emb_params = vocab_size * d_model  # Token embedding (tied with lm_head)
    pos_params = 512 * d_model  # Position embedding
    
    # Per layer: attention (QKV + out) + MLP (up + down) + 2 layer norms
    attn_params = 4 * d_model * d_model  # Q, K, V, O projections
    mlp_params = 2 * d_model * (4 * d_model)  # up + down projection
    norm_params = 2 * d_model * 2  # 2 LayerNorms with weight + bias
    
    layer_params = attn_params + mlp_params + norm_params
    total_layers = layer_params * num_layers
    total = emb_params + pos_params + total_layers + d_model  # + final norm
    
    print(f"GPT: vocab={vocab_size}, d_model={d_model}, heads={num_heads}, layers={num_layers}")
    print(f"  Embeddings:     {format_params(emb_params):>8} ({emb_params/total:.1%})")
    print(f"  Per-layer attn: {format_params(attn_params):>8}")
    print(f"  Per-layer MLP:  {format_params(mlp_params):>8}")
    print(f"  All layers:     {format_params(total_layers):>8} ({total_layers/total:.1%})")
    print(f"  TOTAL:          {format_params(total):>8}")
    print()

# GPT-2 scale
gpt_param_breakdown(50257, 768, 12, 12)   # GPT-2 Small (124M)
gpt_param_breakdown(50257, 1024, 16, 24)  # GPT-2 Medium (350M)
gpt_param_breakdown(50257, 1600, 25, 48)  # GPT-2 XL (1.5B)

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and embed them."""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # Conv2d with kernel_size=stride=patch_size effectively splits into patches
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        # x: [B, C, H, W] → [B, num_patches, d_model]
        x = self.proj(x)             # [B, d_model, H/P, W/P]
        x = x.flatten(2).transpose(1, 2)  # [B, num_patches, d_model]
        return x


class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, d_model=768, num_heads=12, num_layers=12):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        num_patches = self.patch_embed.num_patches
        
        # Learnable [CLS] token and position embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=4*d_model,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        
        # Patch embedding
        x = self.patch_embed(x)  # [B, num_patches, d_model]
        
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)  # [B, 1, d_model]
        x = torch.cat([cls, x], dim=1)  # [B, num_patches+1, d_model]
        
        # Add position embedding
        x = x + self.pos_embed
        
        # Transformer
        x = self.encoder(x)
        x = self.norm(x)
        
        # Classify from CLS token
        return self.head(x[:, 0])


# Build a smaller ViT for demo
vit = ViT(img_size=224, patch_size=16, num_classes=10,
           d_model=192, num_heads=3, num_layers=4)

x = torch.randn(2, 3, 224, 224)
output = vit(x)

print(f"Input: {x.shape}")
print(f"Patches: {vit.patch_embed.num_patches} patches of 16x16")
print(f"Sequence length: {vit.patch_embed.num_patches + 1} (patches + CLS)")
print(f"Output: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in vit.parameters()):,}")

### ViT: How Patch Embedding Works

The key insight of ViT: a 2D convolution with `kernel_size=patch_size` and `stride=patch_size` is equivalent to extracting non-overlapping patches and linearly projecting each one.

In [ ]:
# Demonstrate patch embedding equivalence
img_size, patch_size = 32, 8  # Small image for visualization
channels, d_model = 3, 64
num_patches = (img_size // patch_size) ** 2

img = torch.randn(1, channels, img_size, img_size)

# Method 1: Conv2d (efficient)
conv_embed = nn.Conv2d(channels, d_model, kernel_size=patch_size, stride=patch_size)
patches_conv = conv_embed(img).flatten(2).transpose(1, 2)

# Method 2: Manual unfold + linear (explicit)
patches_manual = img.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
patches_manual = patches_manual.contiguous().view(1, num_patches, channels * patch_size * patch_size)
linear = nn.Linear(channels * patch_size * patch_size, d_model, bias=True)

# Copy conv weights into linear for comparison
linear.weight.data = conv_embed.weight.data.view(d_model, -1)
linear.bias.data = conv_embed.bias.data
patches_linear = linear(patches_manual)

print(f"Image: {img.shape} ({img_size}x{img_size} with {channels} channels)")
print(f"Patch size: {patch_size}x{patch_size}")
print(f"Number of patches: {num_patches} ({img_size//patch_size}x{img_size//patch_size} grid)")
print(f"Patches (conv method): {patches_conv.shape}")
print(f"Patches (manual method): {patches_linear.shape}")
print(f"Results match: {torch.allclose(patches_conv, patches_linear, atol=1e-5)}")

## Architecture Comparison

Let's compare all three architectures side by side.

In [ ]:
# Test ViT with a forward + backward pass
vit_small = ViT(img_size=64, patch_size=8, num_classes=10,
                d_model=128, num_heads=4, num_layers=3)

x = torch.randn(4, 3, 64, 64)
target = torch.randint(0, 10, (4,))

logits = vit_small(x)
loss = F.cross_entropy(logits, target)
loss.backward()

print(f"ViT forward+backward pass:")
print(f"  Image size: 64x64, Patch size: 8x8")
print(f"  Sequence length: {(64//8)**2 + 1} (patches + CLS)")
print(f"  Loss: {loss.item():.4f}")
print(f"  Grad flows: {all(p.grad is not None for p in vit_small.parameters() if p.requires_grad)}")

### ViT Scaling

ViT models are named by their size:

| Model | Layers | d_model | Heads | Params |
|-------|--------|---------|-------|--------|
| ViT-Tiny | 12 | 192 | 3 | 5.7M |
| ViT-Small | 12 | 384 | 6 | 22M |
| ViT-Base | 12 | 768 | 12 | 86M |
| ViT-Large | 24 | 1024 | 16 | 307M |
| ViT-Huge | 32 | 1280 | 16 | 632M |

In [ ]:
# Compare ViT scales
configs = [
    ("ViT-Tiny",  192, 3,  12),
    ("ViT-Small", 384, 6,  12),
    ("ViT-Base",  768, 12, 12),
]

print(f"{'Model':<12} {'d_model':<9} {'Heads':<7} {'Layers':<8} {'Params':<12}")
print("-" * 50)
for name, d, h, l in configs:
    model = ViT(img_size=224, patch_size=16, num_classes=1000,
                d_model=d, num_heads=h, num_layers=l)
    total, _ = count_params(model)
    print(f"{name:<12} {d:<9} {h:<7} {l:<8} {format_params(total):<12}")

## Architecture Comparison: When to Use Each

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def estimate_flops_forward(model, input_tensor):
    """Rough FLOPs estimate via multiply-accumulate counting."""
    # Simple proxy: measure total multiplications
    total_macs = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            total_macs += module.in_features * module.out_features
        elif isinstance(module, nn.Conv2d):
            out_h = input_tensor.shape[2] // (module.stride[0] if hasattr(module, 'stride') else 1)
            out_w = out_h
            total_macs += (module.in_channels * module.kernel_size[0] * module.kernel_size[1] 
                          * module.out_channels * out_h * out_w) // (module.stride[0] ** 2)
    return total_macs * 2  # MACs → FLOPs (multiply + add)

# Compare
models = {
    "ResNet-18": (resnet, torch.randn(1, 3, 224, 224)),
    "GPT (small)": (gpt, torch.randint(0, 256, (1, 128))),
    "ViT (tiny)": (vit, torch.randn(1, 3, 224, 224)),
}

print(f"{'Model':<15} {'Params':>12} {'Input':>25} {'Output':>15}")
print("-" * 70)
for name, (model, inp) in models.items():
    with torch.no_grad():
        out = model(inp)
    params = count_params(model)
    print(f"{name:<15} {params:>12,} {str(tuple(inp.shape)):>25} {str(tuple(out.shape)):>15}")

print("""
Key Design Differences:
┌──────────┬────────────────┬──────────────────┬─────────────────┐
│          │ ResNet         │ GPT              │ ViT             │
├──────────┼────────────────┼──────────────────┼─────────────────┤
│ Input    │ Image pixels   │ Token IDs        │ Image patches   │
│ Core op  │ Convolution    │ Causal attention │ Full attention  │
│ Task     │ Classification │ Generation       │ Classification  │
│ Output   │ Class logits   │ Next-token dist  │ Class logits    │
│ Scaling  │ Depth + width  │ Depth + width    │ Depth + width   │
└──────────┴────────────────┴──────────────────┴─────────────────┘
""")

## GQA Exercise: Understanding the KV Cache Savings

The main motivation for GQA is reducing the KV cache during inference. During autoregressive generation, all previous K and V vectors are cached. With fewer KV heads, the cache is proportionally smaller.

In [ ]:
# KV cache size comparison: MHA vs GQA
def kv_cache_size(seq_len, d_model, num_layers, num_kv_heads, dtype_bytes=2):
    """Calculate KV cache size in bytes."""
    d_k = d_model // num_kv_heads  # This isn't right — d_k is d_model // num_q_heads
    # Actually, KV cache stores [num_kv_heads, seq_len, d_k] per layer
    # For GQA, num_kv_heads < num_q_heads, but d_k = d_model // num_q_heads
    # Total per layer: 2 (K+V) × num_kv_heads × seq_len × d_k × dtype_bytes
    # where d_k = d_model // num_q_heads (fixed regardless of num_kv_heads)
    num_q_heads = 32  # typical
    head_dim = d_model // num_q_heads
    kv_per_layer = 2 * num_kv_heads * seq_len * head_dim * dtype_bytes
    return kv_per_layer * num_layers

# LLaMA-style model: d_model=4096, 32 layers
d_model = 4096
num_layers = 32
seq_len = 4096

print(f"KV Cache Size for seq_len={seq_len}, {num_layers} layers, d_model={d_model}:")
print(f"{'Config':<30} {'KV Heads':<10} {'Cache Size':<15} {'Savings':<10}")
print("-" * 65)

mha_size = kv_cache_size(seq_len, d_model, num_layers, num_kv_heads=32)
for name, num_kv in [("MHA (standard)", 32), ("GQA (LLaMA-2 70B)", 8), ("GQA (Mistral-7B)", 8), ("MQA (single KV head)", 1)]:
    size = kv_cache_size(seq_len, d_model, num_layers, num_kv_heads=num_kv)
    savings = 1 - size / mha_size
    print(f"{name:<30} {num_kv:<10} {size/1e9:.2f} GB       {savings:.0%}")

print("\nFor a batch of 32 sequences, MHA cache = 32GB but GQA(8 heads) = 8GB!")
print("This directly translates to higher serving throughput.")

## Try It Yourself: Add Grouped-Query Attention (GQA) to GPT

**Exercise:** Modify the `CausalSelfAttention` to use **Grouped-Query Attention (GQA)**, where multiple query heads share the same key/value head. This reduces KV cache memory during inference.

In GQA with `num_heads=8` and `num_kv_heads=2`:
- 8 query heads compute independently
- Only 2 KV heads, each shared by 4 query heads
- Saves memory proportional to `num_heads / num_kv_heads`

Hint: Project K and V to fewer heads, then repeat them to match Q's head count.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """GQA: multiple query heads share fewer KV heads."""
    
    def __init__(self, d_model, num_heads, num_kv_heads, max_len=512):
        super().__init__()
        assert num_heads % num_kv_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups = num_heads // num_kv_heads  # queries per KV head
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, num_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        
        mask = torch.tril(torch.ones(max_len, max_len))
        self.register_buffer("mask", mask.view(1, 1, max_len, max_len))
    
    def forward(self, x):
        B, T, _ = x.shape
        
        # Q: full heads, K/V: fewer heads
        q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.num_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_kv_heads, self.d_k).transpose(1, 2)
        
        # Repeat KV heads to match query heads
        # [B, num_kv_heads, T, d_k] → [B, num_heads, T, d_k]
        k = k.repeat_interleave(self.num_groups, dim=1)
        v = v.repeat_interleave(self.num_groups, dim=1)
        
        # Standard attention
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.proj(out)


# Compare parameter counts: MHA vs GQA
d_model = 256
mha = CausalSelfAttention(d_model, num_heads=8)
gqa = GroupedQueryAttention(d_model, num_heads=8, num_kv_heads=2)

x = torch.randn(1, 32, d_model)
mha_out = mha(x)
gqa_out = gqa(x)

print(f"MHA params: {count_params(mha):,} (8 Q heads, 8 KV heads)")
print(f"GQA params: {count_params(gqa):,} (8 Q heads, 2 KV heads)")
print(f"KV param savings: {1 - count_params(gqa)/count_params(mha):.1%}")
print(f"\nBoth produce same output shape: MHA={mha_out.shape}, GQA={gqa_out.shape}")

## Key Takeaways

1. **ResNet** uses skip connections to enable training very deep CNNs — the identity shortcut provides a gradient highway
2. **GPT** is a stack of causal Transformer blocks with weight-tied embeddings — generation is just iterative next-token prediction
3. **ViT** treats images as patch sequences — no convolutions needed, pure attention
4. **Weight tying** (sharing embedding/output weights) saves parameters and improves language models
5. **Causal masking** is the key difference between encoder (BERT/ViT) and decoder (GPT) models
6. **GQA** reduces KV parameters/memory by sharing KV heads across multiple query heads — used in LLaMA 2/3, Mistral
7. All three architectures follow the **same pattern**: embed → process → project to output
8. Scaling any architecture = more layers (depth) + wider hidden dimensions
9. **Patch size** in ViT controls the sequence length vs. patch resolution tradeoff
10. Modern LLMs combine tricks from all: pre-norm (GPT), GQA, RoPE (instead of learned pos embedding), SwiGLU (instead of GELU MLP)